# Initial-State Comparison

This notebook compares different initial states for the same perturbation model and line scan.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

DATA_DIR = Path("data/state_comparison")
plt.rcParams.update({"figure.dpi": 140})

def load_config(dataset_dir):
    with open(dataset_dir / "config.json") as f:
        return json.load(f)

def state_label(config):
    state_type = config["initial_state"]["type"]
    labels = {
        "plus": "plus",
        "ghz": "GHZ",
        "random_product": "random product",
        "chaotic_evolved": "chaotic evolved",
    }
    return labels.get(state_type, state_type)

def load_state_datasets(data_dir):
    candidates = {}
    for dataset_dir in sorted(data_dir.iterdir()) if data_dir.exists() else []:
        if not (dataset_dir / "config.json").exists() or not (dataset_dir / "results.npz").exists():
            continue
        config = load_config(dataset_dir)
        if not str(config.get("experiment_name", "")).startswith("state_comparison"):
            continue
        label = state_label(config)
        item = ("__smoke" in dataset_dir.name, dataset_dir.stat().st_mtime, dataset_dir.name, dataset_dir, config)
        candidates.setdefault(label, []).append(item)
    if not candidates:
        raise FileNotFoundError(f"No state-comparison datasets found in {data_dir}. Run python run_state_comparison.py first.")
    state_data = {}
    for label, items in candidates.items():
        items.sort(key=lambda item: (item[0], -item[1]))
        _, _, name, dataset_dir, config = items[0]
        results = np.load(dataset_dir / "results.npz", allow_pickle=False)
        state_data[label] = (name, config, results)
    return state_data

def get_directional_qfi(data):
    if "qfi_directional" in data.files:
        value = float(data["qfi_directional"])
        if np.isfinite(value) or "qfim" not in data.files:
            return value
    direction = data["direction"]
    qfim = data["qfim"]
    return float(direction @ qfim @ direction)

In [ ]:
state_data = load_state_datasets(DATA_DIR)

for label, (name, config, _) in state_data.items():
    print(label, "dataset=", name, "N=", config["n"], "state=", config["initial_state"]["type"])

## Simulation Error Curves

All curves use the same base Hamiltonian, perturbation basis, time, and line direction. Only the initial state changes.

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.6))
for label, (_, _, data) in state_data.items():
    ax.plot(data["epsilons"], data["exact_error"], marker="o", label=label)

actual_n = next(iter(state_data.values()))[1]["n"]
ax.set_xlabel(r"$\epsilon$ in $\delta = \epsilon(1,0)$")
ax.set_ylabel("simulation error")
ax.set_title(f"Initial-state dependence, N={actual_n}")
ax.legend(frameon=False)
fig.tight_layout()

## Directional QFI

This is the QFI along the scanned perturbation direction, estimated by finite difference.

In [ ]:
labels = []
directional_qfi = []
for label, (_, _, data) in state_data.items():
    labels.append(label)
    directional_qfi.append(get_directional_qfi(data))

fig, ax = plt.subplots(figsize=(5.4, 3.3))
ax.bar(labels, directional_qfi)
ax.set_ylabel(r"$v^T \mathcal{F}_Q v$")
ax.set_title("QFI along the scanned perturbation direction")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()